git add nanoGPT-master/
git commit -m "更新说明"
git push origin main

In [1]:
with open('input.txt','r',encoding = 'utf-8') as f:
    text = f.read()

print('length of dataset in characters: ',len(text))

length of dataset in characters:  1115394


![](images/2026-10-11-10-55-27.png)
#encoding='utf-8' 是一个用来“翻译文本数据”的字符编码规则
![](images/2026-10-11-11-38-40.png)
![](images/2026-10-11-11-41-04.png)
![](images/2026-10-11-11-41-42.png)
![](images/2026-10-11-11-41-56.png)
![](images/2026-10-11-11-42-27.png)
![](images/2026-10-11-11-42-48.png)

In [2]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
#字符编码器与解码器  tokenizer分词器
#把人类看懂的文本（字符 char），转换成神经网络看得懂的数字（整数 integer），以及反向转换回去
#专门构建一个针对该文本的小密码字典，而不是直接用 UTF-8
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l:''.join([itos[i] for i in l])

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


![](images/2026-10-11-15-36-37.png)
![](images/2026-10-11-14-52-46.png)
![](images/2026-10-11-14-54-04.png)
![](images/2026-10-11-14-54-31.png)
![](images/2026-10-11-14-54-42.png)
![](images/2026-10-11-14-54-59.png)
![](images/2026-10-11-15-36-56.png)

![](images/2026-10-11-15-21-28.png)
![](images/2026-10-11-15-27-13.png)
![](images/2026-10-11-15-27-41.png)
![](images/2026-10-11-15-28-07.png)
![](images/2026-10-11-15-29-33.png)
![](images/2026-10-11-15-32-23.png)
![](images/2026-10-11-15-32-40.png)
![](images/2026-10-11-15-33-25.png)

In [4]:
import torch
data = torch.tensor(encode(text),dtype = torch.long)
print(data.shape,data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


![](images/2026-10-11-15-51-16.png)
![](images/2026-10-11-15-51-39.png)
#PyTorch 创建张量时，默认数据类型的确常常是 torch.float32

In [6]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [8]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target:{target}")

when input is tensor([18]) the target:47
when input is tensor([18, 47]) the target:56
when input is tensor([18, 47, 56]) the target:57
when input is tensor([18, 47, 56, 57]) the target:58
when input is tensor([18, 47, 56, 57, 58]) the target:1
when input is tensor([18, 47, 56, 57, 58,  1]) the target:15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target:47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target:58


![](images/2026-10-11-17-39-17.png)
![](images/2026-10-11-17-43-45.png)
![](images/2026-10-11-21-22-31.png)
![](images/2026-10-11-21-22-47.png)
![](images/2026-10-11-21-23-21.png)

In [ ]:
torch.manual_seed(1337)
batch_size = 4  # how many independent sequences will we process in parallel?
block_size = 8  # what is the maximum context length for predictions?

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1,i+block_size] for i in ix])
    return x,y

xb,yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')



![](images/2026-10-11-21-53-54.png)
![](images/2026-10-11-22-16-29.png)
![](images/2026-10-11-22-05-20.png)
![](images/2026-10-11-22-05-37.png)
![](images/2026-10-11-22-06-21.png)
![](images/2026-10-11-22-06-56.png)
![](images/2026-10-11-22-07-11.png)
![](images/2026-10-11-22-08-28.png)
![](images/2026-10-11-22-08-49.png)

![](images/2026-10-11-22-22-47.png)
![](images/2026-10-11-22-37-41.png)
![](images/2026-10-11-22-53-19.png)
